## RAG Application with Langchain, Groq llm and HuggingFace embedding  

In [1]:
# Install the necessary packages
!pip install -qU langchain
!pip install -qU langchain-groq
!pip install -qU langchain-chroma
!pip install -qU langchain_community


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -qU langchain-huggingface
!pip install -qU sentence-transformers
!pip install -qU pypdf


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Initialize the Groq LLM

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load variables from .env
load_dotenv()

# Initialize the Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

#### Initialize Embedding Model

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

#### Load PDF Document

In [17]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Initialize the PyPDFDirectoryLoader with the path to the directory containing PDF files
loader = PyPDFDirectoryLoader("pdf")

# Load the PDF data from the directory
docs = loader.load()

for data in docs:
  print("------------------------")
  print(data.metadata)

------------------------
{'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-30T02:40:41+00:00', 'title': 'FCT Student Handbook Part 1 - Knowledge Base', 'moddate': '2026-09-30T02:40:41+00:00', 'source': 'pdf\\FCT Student Handbook Part 1 - RAG Knowledge Base.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}
------------------------
{'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-30T02:40:41+00:00', 'title': 'FCT Student Handbook Part 1 - Knowledge Base', 'moddate': '2026-09-30T02:40:41+00:00', 'source': 'pdf\\FCT Student Handbook Part 1 - RAG Knowledge Base.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}
------------------------
{'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-09-30T02:40:41+00:00', 'title': 'FCT Student Handbook Part 1 - Knowledge Base', 'moddate': '2026-09-30T02:40:41+00:00', 'source': 'pdf\\FCT Student Handbook Part 1 - RAG Knowledge Base.pdf', 'total_pages': 5, 'page': 2, 'p

In [18]:
len(docs)

20

#### Split Documents into Chunks

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

# Split the documents into chunks
splits = text_splitter.split_documents(docs)

In [20]:
len(splits)

182

#### Create Vector Store and Retriever

In [21]:
from langchain_chroma import Chroma

# Create a vector store from the document chunks
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
     
# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

#### Define Prompt Template

In [22]:
from langchain_core.prompts import ChatPromptTemplate

# Define the system prompt
system_prompt = (
    "You are an intelligent chatbot. "
    "Use the provided PDF context when it is relevant. "
    "If the context is not relevant to the question, answer using your general knowledge. "
    "If you don't know the answer, say that you don't know. "
    "\n\n"
    "{context}"
)

# Create the prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an intelligent chatbot. Use the provided PDF context when it is relevant. If the context is not relevant to the question, answer using your general knowledge. If you don't know the answer, say that you don't know. \n\n{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

#### Retrieval-Augmented Generation (RAG) Chain

In [24]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt)

# Create the RAG chain
rag_chain = create_retrieval_chain(retriever, qa_chain)

#### Examples

In [28]:
response = rag_chain.invoke({"input": "FCT means?"})
response["answer"]

'**FCT** stands for **Faculty of Computing and Technology**. It is the faculty in Sri Lanka that adopts the Learning‑Centered Education (LCE) pedagogy and defines course credits as described in the provided material.'

In [30]:
response = rag_chain.invoke({"input": "Degrees of ICT, CS, ET means?"})
response["answer"]

'**ICT – Information and Communication Technology**  \n- Refers to degree programmes that focus on the design, development, implementation, and management of information systems, networking, software, and communication technologies. In the document you provided, the “Bachelor of Information and Communication Technology Honours Degree (BICT)” is an example of an ICT‑focused qualification.\n\n**CS – Computer Science**  \n- The discipline that studies the theory, design, development, and application of computer systems and software. In the catalogue it appears as “BACHELOR OF SCIENCE HONOURS IN COMPUTER SCIENCE (B.Sc. Hons CS)” with several specialization pathways (Cyber Security, Data Science, Artificial Intelligence, Scientific Computing, etc.).\n\n**ET – Engineering Technology**  \n- A field that applies engineering principles to the practical implementation, operation, and maintenance of technology. The document lists several Engineering‑Technology specialisations, such as:\n  - **ETS

In [31]:
response = rag_chain.invoke({"input": "What is rag?"})
response["answer"]

'**RAG** stands for **Retrieval‑Augmented Generation**.\n\nIt is a technique used in modern AI systems (especially large language models) that combines two steps:\n\n1. **Retrieval** – The model first searches a large external knowledge base (a “corpus”) for pieces of text that are relevant to the user’s query.  \n2. **Generation** – It then feeds those retrieved passages to the language model, which uses them as context to produce its answer.\n\nThe idea is to give the model up‑to‑date, factual information that it might not have memorized during training, while still allowing it to generate fluent, natural‑language responses.  \n\nIn the snippet you saw (“RAG Corpus Document”), the “corpus” refers to the collection of documents that the system can pull from when answering questions. The system first retrieves the most relevant document(s) from that corpus and then generates a response that is grounded in the retrieved material. This approach helps improve accuracy and reduces the chan

### Add Chat History

In [32]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# Define the contextualize system prompt
contextualize_system_prompt = (
    "using chat history and the latest user question, just reformulate question if needed and otherwise return it as is"
)

# Create the contextualize prompt template
contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create the history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

#### Create History-Aware RAG Chain

In [33]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

system_prompt = (
    "You are an intelligent chatbot. Use the following context to answer the question. If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

prompt

ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.

In [34]:
# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt)

# Create the history aware RAG chain
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

#### Manage Chat Session History

In [35]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Initialize the store for session histories
store = {}

# Function to get the session history for a given session ID
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create the conversational RAG chain with session history
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

#### Conversational RAG Chain with Examples

In [39]:
response = conversational_rag_chain.invoke(
    {"input": "BBST?"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'### Bachelor of Biosystems Technology Honours (BBST\u202fHons)\n\n| Item | Details |\n|------|---------|\n| **Full name** | Bachelor of Biosystems Technology Honours (BBST\u202fHons) |\n| **Duration** | 4\u202fyears (8 semesters) |\n| **Annual intake** | 75 students |\n| **Academic structure** | Two semesters per academic year; each semester = 15\u202fweeks of teaching + 2\u202fweeks of study leave |\n| **Year\u202f1** | Common biosystems foundations for all students (core modules in biology, chemistry, mathematics, physics, introductory engineering, and basic computing). |\n| **Specialisations (from Year\u202f2 onward)** | 1. **Biotechnology**  <br>2. **Food Processing Technology**  <br>3. **Environmental Resource Technology** |\n| **Typical credit load** | ~34\u202fcredits per year (example for Year\u202f2: Biostatistics, Bioprocess Technology & Lab I, Environmental Science & Tech, Teaching & Learning Strategies, English for Technology). |\n| **Level\u202f3 & Level\u202f4 modules** 

In [41]:
response = conversational_rag_chain.invoke(
    {"input": "What are the main degree programs?"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'### Main Undergraduate Degree Programs Offered by the Faculty of Computing & Technology (FCT), University of\u202fKelaniya  \n\n| Programme (abbreviation) | Full title | Duration | Typical intake (per year) | Core focus / specialisation |\n|---------------------------|------------|----------|---------------------------|-----------------------------|\n| **BICT** | **Bachelor of Information & Communication Technology Honours** | 4\u202fyears (8 semesters) | 75\u202fstudents | Software development, networking, data communications, ICT project management, and emerging technologies such as AI & IoT. |\n| **BET** | **Bachelor of Engineering Technology Honours** | 4\u202fyears (8 semesters) | 75\u202fstudents | Applied engineering principles, electronics & instrumentation, automation, robotics, and engineering design for industry‑oriented solutions. |\n| **B.Sc (Hons)\u202fCS** | **Bachelor of Science Honours in Computer Science** | 4\u202fyears (8 semesters) | 75\u202fstudents | Foundations

In [43]:
response = conversational_rag_chain.invoke(
    {"input": "List only the names of above 4?"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'- Bachelor of Information & Communication Technology Honours (BICT)  \n- Bachelor of Engineering Technology Honours (BET)  \n- Bachelor of Science Honours in Computer Science (B.Sc\u202f(Hons)\u202fCS)  \n- Bachelor of Biosystems Technology Honours (BBST)'

In [47]:
response = conversational_rag_chain.invoke(
    {"input": "What is last query?"},
    config={"configurable": {"session_id": "103"}},
)
response["answer"]

'I don’t know.'

In [48]:
response = conversational_rag_chain.invoke(
    {"input": "What is last query?"},
    config={"configurable": {"session_id": "101"}},
)
response["answer"]

'Your most recent query was: **“List only the names of above 4?”**'